## Dataset

In [1]:
## Download necessary data files
!mkdir -p data
!cd data
!python -m pip install --user -U gdown
!gdown --folder "https://drive.google.com/drive/folders/1j21cc7-97TedKh_El5E34yI8o5ckI7eK"

# remove the ones we dont need
!rm -rf data/pdbbind_v2016
!rm -rf data/pdbbind_v2020
!rm data/test_set.zip
!rm data/crossdocked_v1.1_rmsd1.0.tar.gz

A subdirectory or file -p already exists.
Error occurred while processing: -p.
A subdirectory or file data already exists.
Error occurred while processing: data.


'gdown' is not recognized as an internal or external command,
operable program or batch file.
'rm' is not recognized as an internal or external command,
operable program or batch file.
'rm' is not recognized as an internal or external command,
operable program or batch file.
'rm' is not recognized as an internal or external command,
operable program or batch file.
'rm' is not recognized as an internal or external command,
operable program or batch file.


## Create Train/Val/Test Split
The provided dataset from DiffDock has only train and test split. We need to make a validation split from train.

In [1]:
import pickle
import random
from pathlib import Path

import lmdb
import torch

lmdb_path = "data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb"
name_split_path = "data/split_by_name.pt"
out_path = "data/crossdocked_pose_split_from_name_val1000.pt"


def pair_key(pf, lf):
    return (str(pf), str(lf))


def pair_key_name(pf, lf):
    return (Path(str(pf)).name, Path(str(lf)).name)


name_split = torch.load(name_split_path, map_location="cpu")
train_pairs = [pair_key(p, l) for p, l in name_split["train"]]
test_pairs = [pair_key(p, l) for p, l in name_split["test"]]

wanted_full = set(train_pairs) | set(test_pairs)
wanted_name = set(pair_key_name(p, l) for p, l in wanted_full)

env = lmdb.open(
    lmdb_path,
    subdir=False,
    readonly=True,
    lock=False,
    readahead=False,
    max_readers=256,
)

mapping = {}
mapping_name = {}

with env.begin() as txn:
    cur = txn.cursor()
    for k, v in cur:
        rec = pickle.loads(v)
        pf = rec.get("protein_filename", "")
        lf = rec.get("ligand_filename", "")
        if not pf or not lf:
            continue

        key_full = pair_key(pf, lf)
        if key_full in wanted_full and key_full not in mapping:
            mapping[key_full] = int(k)

        key_name = pair_key_name(pf, lf)
        if key_name in wanted_name and key_name not in mapping_name:
            mapping_name[key_name] = int(k)

env.close()


def resolve_ids(pairs):
    ids = []
    misses = 0
    for pf, lf in pairs:
        idx = mapping.get((pf, lf))
        if idx is None:
            idx = mapping_name.get(pair_key_name(pf, lf))
        if idx is None:
            misses += 1
            continue
        ids.append(idx)
    return ids, misses


train_ids, train_miss = resolve_ids(train_pairs)
test_ids, test_miss = resolve_ids(test_pairs)

test_set = set(test_ids)
train_ids = [x for x in train_ids if x not in test_set]

seed = 0
val_n = 1000
rng = random.Random(seed)
rng.shuffle(train_ids)

val_ids = train_ids[:val_n]
train_ids = train_ids[val_n:]

if len(val_ids) != val_n:
    raise RuntimeError(f"not enough train ids for val={val_n}: got {len(val_ids)}")

out = {"train": train_ids, "val": val_ids, "test": test_ids}
torch.save(out, out_path)

print(
    "saved:", out_path,
    "\nsizes:", {k: len(out[k]) for k in out},
    "\nmisses:", {"train": train_miss, "test": test_miss},
)

split = torch.load("data/crossdocked_pose_split_from_name_val1000.pt")
assert not (set(split["train"]) & set(split["val"]) &set(split["test"]))

print("no id overlap")

saved: data/crossdocked_pose_split_from_name_val1000.pt 
sizes: {'train': 98990, 'val': 1000, 'test': 100} 
misses: {'train': 10, 'test': 0}
no id overlap


In [2]:
import os
from datamodules import CrossDockedDataModule

lmdb_path = os.path.abspath("data/crossdocked_v1.1_rmsd1.0_pocket10_processed_final.lmdb")
split_path = os.path.abspath("data/crossdocked_pose_split_from_name_val1000.pt")

dm = CrossDockedDataModule(
    lmdb_path=lmdb_path,
    split_pt_path=split_path,
    batch_size=8,
    num_workers=2,
)
dm.setup()

batch = next(iter(dm.train_dataloader()))
print("protein_pos:", batch["protein_pos"].shape)
print("ligand_pos :", batch["ligand_pos"].shape)
print("bond_index :", batch["ligand_bond_index"].shape)
print("bond_type  :", batch["ligand_bond_type"].shape)
print("protein_counts:", batch["protein_counts"][:5].tolist())
print("ligand_counts :", batch["ligand_counts"][:5].tolist())
print("keys[0]:", batch["keys"][0])

protein_pos: torch.Size([3353, 3])
ligand_pos : torch.Size([184, 3])
bond_index : torch.Size([2, 392])
bond_type  : torch.Size([392])
protein_counts: [458, 391, 532, 434, 423]
ligand_counts : [21, 28, 31, 18, 14]
keys[0]: b'175194'


In [ ]:
# finally, we can delete the leftover files we dont need anymore
!rm data/crossdocked_pocket10_pose_split.pt
!rm data/split_by_name.pt